In [4]:
!pip install pycryptodome pillow numpy


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 18.7 MB/s  0:00:00



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from Crypto.Cipher import AES
from PIL import Image
import numpy as np

# 🔑 AES key (16 bytes = 128 bits)
key = b'ThisIsA16ByteKey'

# Convert key to formats
key_hex = key.hex()
key_bin = ''.join(format(byte, '08b') for byte in key)

print("\n🔑 KEY:")
print("HEX :", key_hex)
print("BIN :", key_bin)

# -------------------------------
# Load image
# -------------------------------
img = Image.open("penguin.png")
img = img.convert("RGB")
img.save("penguin.bmp")

data = np.array(img)
shape = data.shape

flat_data = data.flatten()

# -------------------------------
# Padding
# -------------------------------
pad_len = 16 - (len(flat_data) % 16)
if pad_len != 16:
    flat_data = np.concatenate([flat_data, np.zeros(pad_len, dtype=np.uint8)])

# -------------------------------
# AES ECB (Python reference)
# -------------------------------
cipher = AES.new(key, AES.MODE_ECB)

encrypted = bytearray()

# Open file to dump blocks for Verilog
with open("plaintext_blocks.txt", "w") as f_plain, \
     open("ciphertext_blocks.txt", "w") as f_cipher:

    for i in range(0, len(flat_data), 16):
        block = bytes(flat_data[i:i+16])

        # Convert to formats
        block_hex = block.hex()
        block_bin = ''.join(format(b, '08b') for b in block)

        # Write plaintext block
        f_plain.write(block_hex + "\n")

        # Encrypt (Python reference)
        enc_block = cipher.encrypt(block)
        encrypted.extend(enc_block)

        enc_hex = enc_block.hex()
        enc_bin = ''.join(format(b, '08b') for b in enc_block)

        # Write ciphertext block
        f_cipher.write(enc_hex + "\n")

        # (Optional debug print first few blocks)
        if i < 64:
            print(f"\nBlock {i//16}:")
            print("PLAINTEXT HEX:", block_hex)
            print("PLAINTEXT BIN:", block_bin)
            print("CIPHERTEXT HEX:", enc_hex)
            print("CIPHERTEXT BIN:", enc_bin)

# -------------------------------
# Convert encrypted image
# -------------------------------
encrypted = np.frombuffer(encrypted, dtype=np.uint8)
encrypted = encrypted[:shape[0] * shape[1] * 3]
encrypted_img = encrypted.reshape(shape)

enc_image = Image.fromarray(encrypted_img)
enc_image.save("encrypted_ecb.png")

print("\n✅ Files generated:")
print(" - plaintext_blocks.txt (input to Verilog)")
print(" - ciphertext_blocks.txt (expected output)")
print(" - encrypted_ecb.png")

b'ThisIsA16ByteKey'
✅ Done!
📁 Files generated:
 - penguin.bmp (uncompressed reference)
 - encrypted_ecb.png (ECB encrypted image)
